In [2]:
import pandas as pd
import numpy as np
import joblib
import random
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("data\dataset.csv")  # your symptom dataset

# Clean
df = df.dropna(subset=["Disease"])
df = df.fillna("")

df["Disease"] = df["Disease"].str.lower().str.strip()

df.head()

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\adity\AppData\Local\Temp\ipykernel_2964\476436414.py:1: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_csv("data\dataset.csv")  # your symptom dataset


,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,,,,,,,,,,,,,
1,fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,,,,,,,,,,,,,,
2,fungal infection,itching,nodal_skin_eruptions,dischromic _patches,,,,,,,,,,,,,,
3,fungal infection,itching,skin_rash,dischromic _patches,,,,,,,,,,,,,,
4,fungal infection,itching,skin_rash,nodal_skin_eruptions,,,,,,,,,,,,,,


In [4]:
df.shape

(4920, 18)

In [11]:
symptom_cols = [col for col in df.columns if "Symptom" in col]

for col in symptom_cols:
    df[col] = (
        df[col]
        .str.lower()
        .str.strip()
        .str.replace("_", " ")
    )

In [12]:
df = df.drop_duplicates()

In [13]:
all_symptoms = set()

for col in symptom_cols:
    all_symptoms.update(df[col].unique())

all_symptoms.discard("")
all_symptoms = sorted(list(all_symptoms))

In [14]:
augmented_rows = []

for _, row in df.iterrows():
    symptoms = [row[col] for col in symptom_cols if row[col] != ""]

    if len(symptoms) == 0:
        continue

    for _ in range(3):
        subset = random.sample(symptoms, k=random.randint(1, len(symptoms)))

        # 🔥 Add noise (important)
        if random.random() > 0.5:
            subset.append(random.choice(all_symptoms))

        subset = list(set(subset))

        new_row = {"Disease": row["Disease"]}

        for j, s in enumerate(subset):
            new_row[f"Symptom_{j+1}"] = s

        augmented_rows.append(new_row)

aug_df = pd.DataFrame(augmented_rows)

In [15]:
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,fungal infection,itching,skin rash,nodal skin eruptions,dischromic patches,,,,,,,,,,,,,
1,fungal infection,skin rash,nodal skin eruptions,dischromic patches,,,,,,,,,,,,,,
2,fungal infection,itching,nodal skin eruptions,dischromic patches,,,,,,,,,,,,,,
3,fungal infection,itching,skin rash,dischromic patches,,,,,,,,,,,,,,
4,fungal infection,itching,skin rash,nodal skin eruptions,,,,,,,,,,,,,,


In [16]:
df = pd.concat([df, aug_df], ignore_index=True)
df = df.fillna("")
df = df.drop_duplicates()

In [17]:
for i in df.index:
    symptoms = [df.loc[i, col] for col in symptom_cols if df.loc[i, col] != ""]
    random.shuffle(symptoms)

    for j, col in enumerate(symptom_cols):
        df.loc[i, col] = symptoms[j] if j < len(symptoms) else ""

In [18]:
all_symptoms = set()

for col in symptom_cols:
    all_symptoms.update(df[col].unique())

all_symptoms.discard("")
all_symptoms = sorted(list(all_symptoms))

In [19]:
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,fungal infection,itching,nodal skin eruptions,skin rash,dischromic patches,,,,,,,,,,,,,
1,fungal infection,dischromic patches,nodal skin eruptions,skin rash,,,,,,,,,,,,,,
2,fungal infection,dischromic patches,nodal skin eruptions,itching,,,,,,,,,,,,,,
3,fungal infection,skin rash,itching,dischromic patches,,,,,,,,,,,,,,
4,fungal infection,itching,skin rash,nodal skin eruptions,,,,,,,,,,,,,,


In [20]:
X = pd.DataFrame(0, index=df.index, columns=all_symptoms)

for col in symptom_cols:
    for i in df.index:
        symptom = df.loc[i, col]
        if symptom != "":
            X.loc[i, symptom] = 1

y = df["Disease"]

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [45]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8565400843881856


In [46]:
scores = cross_val_score(model, X, y, cv=5)

print("CV Accuracy:", scores.mean())

CV Accuracy: 0.8722663233926913


In [47]:
joblib.dump(model, "ml_models/model1.pkl")
joblib.dump(all_symptoms, "ml_models/symptoms.pkl")

['ml_models/symptoms.pkl']